# LeetCode #1226: Dining Philosophers

https://leetcode.com/problems/the-dining-philosophers/

## Synchronization Approaches

| Approach | Mechanism | Notes |
| :--- | :--- | :--- |
| **Naive: Busy-Wait** | Spin loop | Wastes CPU; prone to livelock |
| **Optimal: Ordered Mutex Acquisition ★** | Two `SemaphoreSlim(1,1)` per chopstick | Pick lower-indexed chopstick first; eliminates circular-wait deadlock |

---

## Understanding the Methods

### Naive: Busy-Wait
Spin in a loop checking a shared flag. Works but burns CPU.

### Optimal: Ordered Mutex Acquisition ★
Each chopstick is guarded by a `SemaphoreSlim(1,1)`. A philosopher always acquires the lower-numbered chopstick before the higher-numbered one. This breaks the circular-wait condition and eliminates deadlock.

**Constraints:**
* `n` philosophers sit at a round table with `n` chopsticks
* `wantsToEat(philosopher, pickLeftFork, pickRightFork, eat, putLeftFork, putRightFork)` is called concurrently
* Each philosopher must pick up both adjacent chopsticks to eat


## Solutions
### C#

In [ ]:
using System;
using System.Threading;

public class DiningPhilosophers
{
    private readonly SemaphoreSlim[] _chopsticks;

    public DiningPhilosophers()
    {
        _chopsticks = new SemaphoreSlim[5];
        for (int i = 0; i < 5; i++)
            _chopsticks[i] = new SemaphoreSlim(1, 1);
    }

    // Ordered acquisition: always grab lower index first to break circular-wait
    public void WantsToEat(int philosopher,
        Action pickLeftFork, Action pickRightFork,
        Action eat,
        Action putLeftFork, Action putRightFork)
    {
        int left  = philosopher;
        int right = (philosopher + 1) % 5;
        int first = Math.Min(left, right);
        int second = Math.Max(left, right);

        _chopsticks[first].Wait();
        _chopsticks[second].Wait();

        pickLeftFork();
        pickRightFork();
        eat();
        putLeftFork();
        putRightFork();

        _chopsticks[second].Release();
        _chopsticks[first].Release();
    }
}

### Python

In [ ]:
import threading

class DiningPhilosophers:
    def __init__(self):
        self.chopsticks = [threading.Lock() for _ in range(5)]

    def wantsToEat(self, philosopher: int,
                   pickLeftFork, pickRightFork,
                   eat,
                   putLeftFork, putRightFork) -> None:
        left  = philosopher
        right = (philosopher + 1) % 5
        first, second = (left, right) if left < right else (right, left)

        with self.chopsticks[first]:
            with self.chopsticks[second]:
                pickLeftFork()
                pickRightFork()
                eat()
                putLeftFork()
                putRightFork()

### Go

In [ ]:
package main

import "sync"

type DiningPhilosophers struct {
	chopsticks [5]sync.Mutex
}

func (d *DiningPhilosophers) WantsToEat(philosopher int,
	pickLeft, pickRight, eat, putLeft, putRight func()) {

	left  := philosopher
	right := (philosopher + 1) % 5
	first, second := left, right
	if left > right {
		first, second = right, left
	}

	d.chopsticks[first].Lock()
	d.chopsticks[second].Lock()

	pickLeft()
	pickRight()
	eat()
	putLeft()
	putRight()

	d.chopsticks[second].Unlock()
	d.chopsticks[first].Unlock()
}

### Rust

In [ ]:
use std::sync::{Arc, Mutex};

struct DiningPhilosophers {
    chopsticks: Vec<Arc<Mutex<()>>>,
}

impl DiningPhilosophers {
    fn new() -> Self {
        DiningPhilosophers {
            chopsticks: (0..5).map(|_| Arc::new(Mutex::new(()))).collect(),
        }
    }

    fn wants_to_eat(
        &self,
        philosopher: usize,
        pick_left: impl Fn(), pick_right: impl Fn(),
        eat: impl Fn(),
        put_left: impl Fn(), put_right: impl Fn(),
    ) {
        let left  = philosopher;
        let right = (philosopher + 1) % 5;
        let (first, second) = if left < right { (left, right) } else { (right, left) };

        let _g1 = self.chopsticks[first].lock().unwrap();
        let _g2 = self.chopsticks[second].lock().unwrap();

        pick_left(); pick_right(); eat(); put_left(); put_right();
    }
}

## Concurrency Scenarios

1. **Circular-wait without ordering**: All 5 philosophers grab their left chopstick simultaneously — deadlock. Ordered acquisition prevents this.
2. **Philosopher 0 and 4 contend for chopstick 0**: Ordered acquisition makes philosopher 4 wait for chopstick 0 before 4, resolving the competition deterministically.
3. **One philosopher eats while others wait**: Philosophers 1, 2, 3 block on chopstick 1 while philosopher 0 eats; they proceed once philosopher 0 releases.
4. **All philosophers alternate eating**: With 5 philosophers and 5 chopsticks, at most 2 can eat simultaneously; ordering ensures fairness over time.
5. **Philosopher starved by neighbors**: If philosophers 0 and 2 eat repeatedly, philosopher 1 (sharing chopsticks with both) must wait; the mutex ensures eventual progress.
